# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/home/whale/.venvs/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-05-18 13:11:58.719067: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-18 13:11:58.722904: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-18 13:11:58.782298: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-18 13:11:58.782348: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-18 13:11:58.782389: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# Prepare feature matrix X and label vector y
X = df.drop(columns=["Class"]).values
y = df["Class"].values


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

base_model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

base_model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

base_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = base_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=2
)

Epoch 1/20
13/13 - 1s - loss: 1.1331 - accuracy: 0.2525 - val_loss: 0.7455 - val_accuracy: 0.8000 - 1s/epoch - 88ms/step
Epoch 2/20
13/13 - 0s - loss: 0.7512 - accuracy: 0.7172 - val_loss: 0.4952 - val_accuracy: 1.0000 - 98ms/epoch - 8ms/step
Epoch 3/20
13/13 - 0s - loss: 0.5139 - accuracy: 0.8990 - val_loss: 0.3382 - val_accuracy: 1.0000 - 80ms/epoch - 6ms/step
Epoch 4/20
13/13 - 0s - loss: 0.3615 - accuracy: 0.9798 - val_loss: 0.2281 - val_accuracy: 1.0000 - 86ms/epoch - 7ms/step
Epoch 5/20
13/13 - 0s - loss: 0.2541 - accuracy: 0.9899 - val_loss: 0.1573 - val_accuracy: 1.0000 - 86ms/epoch - 7ms/step
Epoch 6/20
13/13 - 0s - loss: 0.1821 - accuracy: 0.9899 - val_loss: 0.1193 - val_accuracy: 1.0000 - 81ms/epoch - 6ms/step
Epoch 7/20
13/13 - 0s - loss: 0.1334 - accuracy: 0.9899 - val_loss: 0.0915 - val_accuracy: 1.0000 - 93ms/epoch - 7ms/step
Epoch 8/20
13/13 - 0s - loss: 0.1017 - accuracy: 0.9899 - val_loss: 0.0732 - val_accuracy: 1.0000 - 80ms/epoch - 6ms/step
Epoch 9/20
13/13 - 0s - l

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

test_loss, test_acc = base_model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

preds = base_model.predict(X_test_scaled)
y_pred = np.argmax(preds, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))
print("Confusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Test accuracy: 0.9815
2/2 [==============================] - 0s 5ms/step

Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix:

[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

import os

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(base_model)
tflite_model = converter.convert()

fname = 'model_base.tflite'
with open(fname, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(fname) / 1024.0
print(f"Saved TFLite model to {fname} ({size_kb:.2f} KB)")


INFO:tensorflow:Assets written to: /tmp/tmpgchvnck_/assets


INFO:tensorflow:Assets written to: /tmp/tmpgchvnck_/assets


Saved TFLite model to model_base.tflite (14.07 KB)


2026-05-18 13:12:31.726437: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 13:12:31.726521: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 13:12:31.727500: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpgchvnck_
2026-05-18 13:12:31.729502: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 13:12:31.729536: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpgchvnck_
2026-05-18 13:12:31.734545: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-18 13:12:31.736185: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 13:12:31.810268: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpgchvnck_
2026-05

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_test, num_samples=100)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        # Force input and output tensors to int8
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # Dynamic range quantization: weights are quantized, activations remain float
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.
    tflite_model = converter.convert()
    with open(filename, 'wb') as f:
        f.write(tflite_model)

    # Helper function to compute file size in KB
    def file_size_kb_local(p):
        import os
        return os.path.getsize(p) / 1024.0

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    y_pred = []
    for i in range(len(X_test)):
        input_data = X_test[i:i+1].astype(np.float32)

        # Handle quantized input if necessary
        in_det = input_details[0]
        if in_det['dtype'] == np.int8 or in_det['dtype'] == np.uint8:
            scale, zero_point = in_det['quantization']
            if scale == 0:
                # fallback: cast to expected dtype
                q_input = input_data.astype(in_det['dtype'])
            else:
                q_input = np.round(input_data / scale + zero_point).astype(in_det['dtype'])
            interpreter.set_tensor(in_det['index'], q_input)
        else:
            # float input
            interpreter.set_tensor(in_det['index'], input_data.astype(in_det['dtype']))

        interpreter.invoke()

        out_det = output_details[0]
        output_data = interpreter.get_tensor(out_det['index'])

        # Dequantize output if necessary
        if out_det['dtype'] == np.int8 or out_det['dtype'] == np.uint8:
            scale, zero_point = out_det['quantization']
            if scale == 0:
                dequant = output_data.astype(np.float32)
            else:
                dequant = (output_data.astype(np.float32) - zero_point) * scale
        else:
            dequant = output_data.astype(np.float32)

        y_pred.append(np.argmax(dequant, axis=1)[0])

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb_local(filename):.2f} KB")
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred))
    print("Confusion Matrix:\n")
    print(confusion_matrix(y_true, y_pred))


In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quantize_and_evaluate(base_model, X_test_scaled, y_test_cat, 'int8', 'model_int8.tflite')
quantize_and_evaluate(base_model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')
quantize_and_evaluate(base_model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')


INFO:tensorflow:Assets written to: /tmp/tmp20nefo0e/assets


INFO:tensorflow:Assets written to: /tmp/tmp20nefo0e/assets
/home/whale/.venvs/tinyml-arduino/lib64/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-18 13:12:36.952555: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 13:12:36.952602: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 13:12:36.952932: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp20nefo0e
2026-05-18 13:12:36.954694: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 13:12:36.954718: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp20nefo0e
2026-05-18 13:12:36.959329: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-0


INT8 TFLite model size: 5.74 KB

Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix:

[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /tmp/tmp5c2na9bv/assets


INFO:tensorflow:Assets written to: /tmp/tmp5c2na9bv/assets
2026-05-18 13:12:37.944336: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 13:12:37.944382: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 13:12:37.944673: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp5c2na9bv
2026-05-18 13:12:37.946325: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 13:12:37.946346: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp5c2na9bv
2026-05-18 13:12:37.950905: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 13:12:38.012649: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp5c2na9bv
2026-05-18 13:12:38.028858: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


FLOAT16 TFLite model size: 8.95 KB

Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix:

[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /tmp/tmpu_rpjpev/assets


INFO:tensorflow:Assets written to: /tmp/tmpu_rpjpev/assets



DYNAMIC TFLite model size: 8.17 KB

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix:

[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


2026-05-18 13:12:38.821763: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 13:12:38.821805: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 13:12:38.822112: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpu_rpjpev
2026-05-18 13:12:38.823418: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 13:12:38.823441: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpu_rpjpev
2026-05-18 13:12:38.826635: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 13:12:38.892575: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpu_rpjpev
2026-05-18 13:12:38.912663: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 90550 m

## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# Configure pruning schedule
batch_size_prune = 8
prune_epochs = 10
train_size = X_train_scaled.shape[0]
# total training steps = steps per epoch * epochs
steps_per_epoch = int(np.ceil(train_size / batch_size_prune))
end_step = steps_per_epoch * prune_epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    prune_low_magnitude(Dense(64, activation='relu', input_shape=(num_features,)), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(32, activation='relu'), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(num_classes, activation='softmax'), pruning_schedule=pruning_schedule)
])

pruned_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 3 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 4 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 5 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

history_pruned = pruned_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=prune_epochs,
    batch_size=batch_size_prune,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=2
)

Epoch 1/10
13/13 - 3s - loss: 0.9057 - accuracy: 0.6869 - val_loss: 0.7948 - val_accuracy: 0.8800 - 3s/epoch - 231ms/step
Epoch 2/10
13/13 - 0s - loss: 0.6584 - accuracy: 0.9394 - val_loss: 0.6015 - val_accuracy: 0.9200 - 86ms/epoch - 7ms/step
Epoch 3/10
13/13 - 0s - loss: 0.4785 - accuracy: 0.9596 - val_loss: 0.4551 - val_accuracy: 0.9200 - 92ms/epoch - 7ms/step
Epoch 4/10
13/13 - 0s - loss: 0.3426 - accuracy: 0.9697 - val_loss: 0.3477 - val_accuracy: 0.9200 - 90ms/epoch - 7ms/step
Epoch 5/10
13/13 - 0s - loss: 0.2429 - accuracy: 0.9899 - val_loss: 0.2739 - val_accuracy: 0.9200 - 92ms/epoch - 7ms/step
Epoch 6/10
13/13 - 0s - loss: 0.1760 - accuracy: 0.9899 - val_loss: 0.2367 - val_accuracy: 0.9200 - 94ms/epoch - 7ms/step
Epoch 7/10
13/13 - 0s - loss: 0.1308 - accuracy: 0.9899 - val_loss: 0.2091 - val_accuracy: 0.9200 - 91ms/epoch - 7ms/step
Epoch 8/10
13/13 - 0s - loss: 0.1776 - accuracy: 0.9899 - val_loss: 0.3194 - val_accuracy: 0.9600 - 109ms/epoch - 8ms/step
Epoch 9/10
13/13 - 0s -

In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# Strip pruning wrappers before exporting
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

# Convert stripped model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
tflite_pruned = converter.convert()
pruned_fname = 'model_pruned.tflite'
with open(pruned_fname, 'wb') as f:
    f.write(tflite_pruned)

print(f"Saved pruned TFLite model to {pruned_fname} ({os.path.getsize(pruned_fname)/1024.0:.2f} KB)")


INFO:tensorflow:Assets written to: /tmp/tmp2xwo_oiu/assets


INFO:tensorflow:Assets written to: /tmp/tmp2xwo_oiu/assets
2026-05-18 13:12:52.777843: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 13:12:52.777915: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.


Saved pruned TFLite model to model_pruned.tflite (14.14 KB)


2026-05-18 13:12:52.778461: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp2xwo_oiu
2026-05-18 13:12:52.779828: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 13:12:52.779849: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp2xwo_oiu
2026-05-18 13:12:52.783293: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 13:12:52.815151: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp2xwo_oiu
2026-05-18 13:12:52.827020: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 48593 microseconds.


In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

preds_pruned = stripped_pruned_model.predict(X_test_scaled)
y_pred_pruned = np.argmax(preds_pruned, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print("\nPruned model evaluation:\n")
print(classification_report(y_true, y_pred_pruned))
print("Confusion Matrix:\n")
print(confusion_matrix(y_true, y_pred_pruned))

2/2 [==============================] - 0s 6ms/step

Pruned model evaluation:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix:

[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 32)                448       
                                                                 
 dense_7 (Dense)             (None, 16)                528       
                                                                 
 dense_8 (Dense)             (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = base_model.predict(X_train_scaled)

4/4 [==============================] - 0s 4ms/step


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# Create combined labels: [hard_labels | soft_labels]
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)

def distillation_loss(y_true_combined, y_pred):
    # Split combined labels into hard and soft targets
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]

    # Compute categorical crossentropy for hard and soft labels
    loss_hard = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    loss_soft = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    alpha = 0.5
    return alpha * loss_hard + (1.0 - alpha) * loss_soft

def distillation_loss(y_true_combined, y_pred):

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    # (Implementation moved up where combined labels were created.)
    # This placeholder will not be used because a concrete implementation
    # was defined in the previous cell.
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]
    loss_hard = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    loss_soft = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)
    alpha = 0.5
    return alpha * loss_hard + (1.0 - alpha) * loss_soft

In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# Define a metric that uses the hard labels portion for accuracy
def hard_accuracy(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :num_classes]
    return tf.keras.metrics.categorical_accuracy(y_true_hard, y_pred)

student_model.compile(optimizer='adam', loss=distillation_loss, metrics=[hard_accuracy])

history_kd = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=2
)

Epoch 1/10
13/13 - 1s - loss: 1.2125 - hard_accuracy: 0.2121 - val_loss: 0.9837 - val_hard_accuracy: 0.5600 - 1s/epoch - 113ms/step
Epoch 2/10
13/13 - 0s - loss: 1.0404 - hard_accuracy: 0.4141 - val_loss: 0.8904 - val_hard_accuracy: 0.6400 - 97ms/epoch - 7ms/step
Epoch 3/10
13/13 - 0s - loss: 0.9378 - hard_accuracy: 0.5657 - val_loss: 0.8056 - val_hard_accuracy: 0.8800 - 96ms/epoch - 7ms/step
Epoch 4/10
13/13 - 0s - loss: 0.8498 - hard_accuracy: 0.7778 - val_loss: 0.7434 - val_hard_accuracy: 0.9200 - 90ms/epoch - 7ms/step
Epoch 5/10
13/13 - 0s - loss: 0.7716 - hard_accuracy: 0.8485 - val_loss: 0.6738 - val_hard_accuracy: 0.9600 - 94ms/epoch - 7ms/step
Epoch 6/10
13/13 - 0s - loss: 0.6974 - hard_accuracy: 0.8889 - val_loss: 0.6068 - val_hard_accuracy: 0.9600 - 94ms/epoch - 7ms/step
Epoch 7/10
13/13 - 0s - loss: 0.6168 - hard_accuracy: 0.9091 - val_loss: 0.5342 - val_hard_accuracy: 0.9600 - 96ms/epoch - 7ms/step
Epoch 8/10
13/13 - 0s - loss: 0.5304 - hard_accuracy: 0.9293 - val_loss: 0.4

In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# Convert student model to TFLite and save
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd = converter.convert()
kdfname = 'model_kd.tflite'
with open(kdfname, 'wb') as f:
    f.write(tflite_kd)

print(f"Saved knowledge-distilled TFLite model to {kdfname} ({os.path.getsize(kdfname)/1024.0:.2f} KB)")


INFO:tensorflow:Assets written to: /tmp/tmpbhldp4fx/assets


INFO:tensorflow:Assets written to: /tmp/tmpbhldp4fx/assets


Saved knowledge-distilled TFLite model to model_kd.tflite (6.10 KB)


2026-05-18 13:13:15.087411: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 13:13:15.087459: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 13:13:15.087766: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpbhldp4fx
2026-05-18 13:13:15.089094: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 13:13:15.089114: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpbhldp4fx
2026-05-18 13:13:15.092835: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 13:13:15.165775: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpbhldp4fx
2026-05-18 13:13:15.185847: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 98080 m

In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

preds_kd = student_model.predict(X_test_scaled)
y_pred_kd = np.argmax(preds_kd, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print("\nKnowledge-distilled student model evaluation:\n")
print(classification_report(y_true, y_pred_kd))
print("Confusion Matrix:\n")
print(confusion_matrix(y_true, y_pred_kd))

2/2 [==============================] - 0s 5ms/step

Knowledge-distilled student model evaluation:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.95      0.95        21
           2       1.00      0.93      0.97        15

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion Matrix:

[[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [28]:
# <-- (if needed) Enter your code here -->
import os

# Step 1: Compare sizes of previously generated TFLite models (if present)
tflite_files = [
    'model_base.tflite',
    'model_int8.tflite',
    'model_float16.tflite',
    'model_dynamic.tflite',
    'model_pruned.tflite',
    'model_kd.tflite'
]

print("Existing TFLite files and sizes (KB):")
existing_sizes = {}
for f in tflite_files:
    if os.path.exists(f):
        existing_sizes[f] = os.path.getsize(f) / 1024.0
        print(f" - {f}: {existing_sizes[f]:.2f} KB")
    else:
        print(f" - {f}: (not found)")

# Quick performance summary from previously evaluated Keras models
def keras_eval(model, X, y_cat, name):
    loss, acc = model.evaluate(X, y_cat, verbose=0)
    print(f"{name}: loss={loss:.4f}, acc={acc:.4f}")
    return acc

print("\nKeras model test performance:")
base_acc = keras_eval(base_model, X_test_scaled, y_test_cat, 'Base model')
pruned_acc = None
try:
    pruned_acc = keras_eval(stripped_pruned_model, X_test_scaled, y_test_cat, 'Pruned (stripped) model')
except Exception:
    pass
# The student model was trained with combined labels (hard + soft). Prepare combined test labels
teacher_preds_test = base_model.predict(X_test_scaled)
y_test_combined = np.concatenate([y_test_cat, teacher_preds_test], axis=1)
kd_acc = keras_eval(student_model, X_test_scaled, y_test_combined, 'Student (KD) model')

# Strategy: start from the small student model (KD) and apply post-training quantization
# (dynamic and int8) to try to reduce size further while retaining accuracy.

print("\nAttempting further reduction: quantize the student (KD) model")
quantize_and_evaluate(student_model, X_test_scaled, y_test_cat, 'dynamic', 'model_kd_dynamic.tflite')
quantize_and_evaluate(student_model, X_test_scaled, y_test_cat, 'float16', 'model_kd_float16.tflite')
quantize_and_evaluate(student_model, X_test_scaled, y_test_cat, 'int8', 'model_kd_int8.tflite')

# Optionally, attempt lightweight pruning of the student and then quantize.
print("\nAttempting light pruning + quantization on student model (few epochs to fine-tune)")
prune_sched_small = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.2,
    final_sparsity=0.5,
    begin_step=0,
    end_step=int(np.ceil(X_train_scaled.shape[0] / 8) * 4)
)

pruned_student = Sequential([
    tfmot.sparsity.keras.prune_low_magnitude(Dense(32, activation='relu', input_shape=(num_features,)), pruning_schedule=prune_sched_small),
    tfmot.sparsity.keras.prune_low_magnitude(Dense(16, activation='relu'), pruning_schedule=prune_sched_small),
    tfmot.sparsity.keras.prune_low_magnitude(Dense(num_classes, activation='softmax'), pruning_schedule=prune_sched_small)
])

# Initialize pruned_student weights from trained student_model where shapes match
try:
    pruned_student.build((None, num_features))
    # Extract weights from student_model and set to pruned_student where possible
    s_weights = student_model.get_weights()
    # pruned_student has extra pruning variables; set layer-wise weights manually
    # Map Dense layers (kernel and bias) from student_model into pruned_student underlying layers
    idx = 0
    for layer in pruned_student.layers:
        # pruning wrapper: layer.layer is the wrapped Dense
        inner = getattr(layer, 'layer', layer)
        weights = inner.get_weights()
        if len(weights) == 0:
            continue
        # replace with corresponding weights from student_model
        w = s_weights[idx]
        b = s_weights[idx+1]
        inner.set_weights([w, b])
        idx += 2
except Exception as e:
    print("Warning: could not initialize pruned_student weights from student_model:", e)

pruned_student.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]
try:
    pruned_student.fit(X_train_scaled, y_train_cat, epochs=4, batch_size=8, validation_split=0.2, callbacks=callbacks, verbose=2)
    stripped_pruned_student = tfmot.sparsity.keras.strip_pruning(pruned_student)
    # Quantize stripped pruned student (dynamic and int8)
    quantize_and_evaluate(stripped_pruned_student, X_test_scaled, y_test_cat, 'dynamic', 'model_pruned_student_dynamic.tflite')
    quantize_and_evaluate(stripped_pruned_student, X_test_scaled, y_test_cat, 'int8', 'model_pruned_student_int8.tflite')
except Exception as e:
    print("Pruning+fine-tune step failed or skipped:", e)

# Summarize results: list all tflite files and sizes after our operations
print("\nFinal TFLite files and sizes (KB):")
for f in sorted([p for p in os.listdir('.') if p.endswith('.tflite')]):
    print(f" - {f}: {os.path.getsize(f)/1024.0:.2f} KB")

Existing TFLite files and sizes (KB):
 - model_base.tflite: 14.07 KB
 - model_int8.tflite: 5.74 KB
 - model_float16.tflite: 8.95 KB
 - model_dynamic.tflite: 8.17 KB
 - model_pruned.tflite: 14.14 KB
 - model_kd.tflite: 6.10 KB

Keras model test performance:
Base model: loss=0.0689, acc=0.9815
2/2 [==============================] - 0s 3ms/step
Student (KD) model: loss=0.3361, acc=0.9630

Attempting further reduction: quantize the student (KD) model
INFO:tensorflow:Assets written to: /tmp/tmp7k5jgz9z/assets


INFO:tensorflow:Assets written to: /tmp/tmp7k5jgz9z/assets
2026-05-18 13:27:55.929767: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 13:27:55.929817: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 13:27:55.930134: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp7k5jgz9z
2026-05-18 13:27:55.931786: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 13:27:55.931807: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp7k5jgz9z
2026-05-18 13:27:55.936666: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 13:27:56.014105: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp7k5jgz9z
2026-05-18 13:27:56.044030: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


DYNAMIC TFLite model size: 6.11 KB

Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.95      0.95        21
           2       1.00      0.93      0.97        15

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion Matrix:

[[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]
INFO:tensorflow:Assets written to: /tmp/tmplh7agj6d/assets


INFO:tensorflow:Assets written to: /tmp/tmplh7agj6d/assets
2026-05-18 13:27:57.495664: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 13:27:57.495801: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 13:27:57.496351: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmplh7agj6d
2026-05-18 13:27:57.499283: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 13:27:57.499344: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmplh7agj6d
2026-05-18 13:27:57.506966: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 13:27:57.607199: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmplh7agj6d
2026-05-18 13:27:57.635975: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


FLOAT16 TFLite model size: 5.00 KB

Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.95      0.95        21
           2       1.00      0.93      0.97        15

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion Matrix:

[[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]
INFO:tensorflow:Assets written to: /tmp/tmpgnkkuwvb/assets


...


INT8 TFLite model size: 3.62 KB

Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.95      0.95        21
           2       1.00      0.93      0.97        15

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion Matrix:

[[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]

Attempting light pruning + quantization on student model (few epochs to fine-tune)
Epoch 1/4
13/13 - 2s - loss: 0.2843 - accuracy: 0.9697 - val_loss: 0.2611 - val_accuracy: 0.9600 - 2s/epoch - 191ms/step
Epoch 2/4
13/13 - 0s - loss: 0.2223 - accuracy: 0.9798 - val_loss: 0.2162 - val_accuracy: 0.9600 - 93ms/epoch - 7ms/step
Epoch 3/4
13/13 - 0s - loss: 0.1782 - accuracy: 0.9899 - val_loss: 0.1880 - val_accuracy: 0.9600 - 106ms/epoch - 8ms/step
Epoch 4/4
13/13 - 0s - loss: 0.1433 - accuracy: 0.9899 - 

INFO:tensorflow:Assets written to: /tmp/tmp5ziztw8q/assets
2026-05-18 13:28:02.815003: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 13:28:02.815052: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 13:28:02.815328: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp5ziztw8q
2026-05-18 13:28:02.816449: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 13:28:02.816467: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp5ziztw8q
2026-05-18 13:28:02.818785: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 13:28:02.845776: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp5ziztw8q
2026-05-18 13:28:02.856570: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


DYNAMIC TFLite model size: 6.18 KB

Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.95      0.95        21
           2       1.00      0.93      0.97        15

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion Matrix:

[[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]
INFO:tensorflow:Assets written to: /tmp/tmpgrcbynzi/assets


INFO:tensorflow:Assets written to: /tmp/tmpgrcbynzi/assets
/home/whale/.venvs/tinyml-arduino/lib64/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-18 13:28:03.468395: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 13:28:03.468441: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 13:28:03.468743: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpgrcbynzi
2026-05-18 13:28:03.469898: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 13:28:03.469918: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpgrcbynzi
2026-05-18 13:28:03.472428: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-0


INT8 TFLite model size: 3.71 KB

Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.95      0.95        21
           2       1.00      0.93      0.97        15

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion Matrix:

[[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]

Final TFLite files and sizes (KB):
 - model_base.tflite: 14.07 KB
 - model_dynamic.tflite: 8.17 KB
 - model_float16.tflite: 8.95 KB
 - model_int8.tflite: 5.74 KB
 - model_kd.tflite: 6.10 KB
 - model_kd_dynamic.tflite: 6.11 KB
 - model_kd_float16.tflite: 5.00 KB
 - model_kd_int8.tflite: 3.62 KB
 - model_pruned.tflite: 14.14 KB
 - model_pruned_student_dynamic.tflite: 6.18 KB
 - model_pruned_student_int8.tflite: 3.71 KB


# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
